In [20]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
from qiskit import transpile

def marked_state_oracle(marked: str) -> QuantumCircuit:
    qc = QuantumCircuit(2)
    reversed_marked = marked[::-1]
    for i, bit in enumerate(reversed_marked):
        if bit == '0':
            qc.x(i)
    qc.cz(0, 1)
    for i, bit in enumerate(reversed_marked):
        if bit == '0':
            qc.x(i)
    return qc

def diffusion_operator() -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.h([0, 1])
    qc.append(marked_state_oracle('00').to_gate(), [0, 1])
    qc.h([0, 1])
    return qc

def build_grover_circuit(marked: str, iterations: int = 1) -> QuantumCircuit:
    qc = QuantumCircuit(2, 2)
    qc.h([0, 1])

    oracle = marked_state_oracle(marked)
    diffusion = diffusion_operator()

    for _ in range(iterations):
        qc.append(oracle.to_gate(), [0, 1])
        qc.append(diffusion.to_gate(), [0, 1])

    qc.measure([0, 1], [0, 1])
    return qc

grover_circuit = build_grover_circuit('11', iterations=1)
print(grover_circuit.draw())



sim = AerSimulator()
compiled = transpile(grover_circuit, sim)
job = sim.run(compiled, shots=1024)
counts = job.result().get_counts(compiled)
print(counts)
plot_histogram(counts)
plt.show()

     ┌───┐┌──────────────┐┌──────────────┐┌─┐   
q_0: ┤ H ├┤0             ├┤0             ├┤M├───
     ├───┤│  circuit-142 ││  circuit-143 │└╥┘┌─┐
q_1: ┤ H ├┤1             ├┤1             ├─╫─┤M├
     └───┘└──────────────┘└──────────────┘ ║ └╥┘
c: 2/══════════════════════════════════════╩══╩═
                                           0  1 
{'11': 1024}
